# 04 Classifier Experiments

Notebook-native classifier experiments. This notebook replaces `train_classifier.py` and the old classifier grid script.

In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import time

import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, DataLoader, Subset
from torchvision import datasets, transforms

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from config import Config
from models.classifier import FruitCNN

ROOT


In [ ]:
def get_transform(img_size, train=True, augmentation_policy="none"):
    tf_list = [transforms.Resize((img_size, img_size))]
    if train and augmentation_policy == "classical":
        tf_list.extend([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ])
    tf_list.extend([
        transforms.ToTensor(),
        transforms.Normalize([0.5] * 3, [0.5] * 3),
    ])
    return transforms.Compose(tf_list)


def scenario_augmentation_policy(scenario):
    return "classical" if scenario == "real_aug" else "none"


def subsample_dataset(dataset, n_per_class, seed=42):
    from collections import defaultdict
    rng = random.Random(seed)
    class_indices = defaultdict(list)
    for idx, (_, label) in enumerate(dataset.samples):
        class_indices[label].append(idx)
    selected = []
    for label, indices in sorted(class_indices.items()):
        rng.shuffle(indices)
        selected.extend(indices[:n_per_class])
    return Subset(dataset, selected)


def build_dataset(cfg, scenario, n_per_class, synth_dir="data_synth", real_train_root=None, test_root=None):
    augmentation_policy = scenario_augmentation_policy(scenario)
    tf_train = get_transform(cfg.img_size, train=True, augmentation_policy=augmentation_policy)
    tf_test = get_transform(cfg.img_size, train=False, augmentation_policy="none")

    real_train_root = Path(real_train_root) if real_train_root else cfg.data_root / "train"
    test_root = Path(test_root) if test_root else cfg.data_root / "test"

    real_train = datasets.ImageFolder(str(real_train_root), transform=tf_train)
    test_ds = datasets.ImageFolder(str(test_root), transform=tf_test)

    if scenario in {"real", "real_aug"}:
        train_ds = subsample_dataset(real_train, n_per_class, cfg.seed) if n_per_class else real_train
    elif scenario == "synth":
        synth_train = datasets.ImageFolder(str(synth_dir), transform=tf_train)
        train_ds = subsample_dataset(synth_train, n_per_class, cfg.seed) if n_per_class else synth_train
    elif scenario == "both":
        synth_train = datasets.ImageFolder(str(synth_dir), transform=tf_train)
        if n_per_class:
            real_sub = subsample_dataset(real_train, n_per_class, cfg.seed)
            synth_sub = subsample_dataset(synth_train, n_per_class, cfg.seed)
            train_ds = ConcatDataset([real_sub, synth_sub])
        else:
            train_ds = ConcatDataset([real_train, synth_train])
    else:
        raise ValueError(f"Unknown scenario: {scenario}")

    return train_ds, test_ds, real_train.classes, augmentation_policy


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
    return all_preds, all_labels


def run_classifier_notebook(cfg, scenario, n_per_class, synth_dir, out_dir, real_train_root=None, test_root=None, time_breakdown=None, extra_metadata=None):
    from sklearn.metrics import classification_report

    device = torch.device(cfg.resolve_device())
    torch.manual_seed(cfg.seed)
    random.seed(cfg.seed)

    train_ds, test_ds, class_names, augmentation_policy = build_dataset(
        cfg,
        scenario,
        n_per_class,
        synth_dir=synth_dir,
        real_train_root=real_train_root,
        test_root=test_root,
    )

    loader_kwargs = cfg.loader_options()
    train_loader = DataLoader(train_ds, batch_size=cfg.clf_batch, shuffle=True, drop_last=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, batch_size=cfg.clf_batch, shuffle=False, **loader_kwargs)

    model = FruitCNN(num_classes=len(class_names)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.clf_lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.clf_epochs)

    print(f"[{scenario}] n_per_class={n_per_class or 'all'} train_size={len(train_ds)} device={device}")
    t0 = time.time()
    for epoch in range(1, cfg.clf_epochs + 1):
        loss, acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        scheduler.step()
        if epoch % 10 == 0 or epoch == cfg.clf_epochs:
            print(f"  Epoch {epoch:03d}  loss={loss:.4f}  train_acc={acc:.4f}")
    train_time = time.time() - t0

    preds, labels = evaluate(model, test_loader, device)
    report = classification_report(labels, preds, target_names=class_names, output_dict=True)
    test_acc = report["accuracy"]

    time_breakdown = time_breakdown or {}
    gan_train_time = round(float(time_breakdown.get("gan_train_time_sec", 0.0)), 1)
    synth_generation_time = round(float(time_breakdown.get("synth_generation_time_sec", 0.0)), 1)
    pipeline_time = round(train_time + gan_train_time + synth_generation_time, 1)

    result = {
        "scenario": scenario,
        "augmentation_policy": augmentation_policy,
        "n_per_class": n_per_class or "all",
        "train_size": len(train_ds),
        "test_accuracy": round(test_acc, 4),
        "train_time_sec": round(train_time, 1),
        "classifier_train_time_sec": round(train_time, 1),
        "gan_train_time_sec": gan_train_time,
        "synth_generation_time_sec": synth_generation_time,
        "pipeline_time_sec": pipeline_time,
        "real_train_root": str(real_train_root or (cfg.data_root / "train")),
        "test_root": str(test_root or (cfg.data_root / "test")),
        "synth_dir": str(synth_dir),
        "per_class": {
            name: {
                "precision": round(report[name]["precision"], 4),
                "recall": round(report[name]["recall"], 4),
                "f1": round(report[name]["f1-score"], 4),
            }
            for name in class_names
        },
    }
    if extra_metadata:
        result.update(extra_metadata)

    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    tag = f"{scenario}_n{n_per_class}" if n_per_class else f"{scenario}_all"
    with open(out_path / f"result_{tag}.json", "w") as f:
        json.dump(result, f, indent=2)
    return result


def run_grid_notebook(cfg, sizes, scenarios, synth_dir, out_dir, real_train_root=None, test_root=None):
    all_results = []
    for n in sizes:
        for scenario in scenarios:
            result = run_classifier_notebook(
                cfg,
                scenario,
                n,
                synth_dir,
                out_dir,
                real_train_root=real_train_root,
                test_root=test_root,
            )
            all_results.append(result)
    out_path = Path(out_dir)
    with open(out_path / "all_results.json", "w") as f:
        json.dump(all_results, f, indent=2)
    return all_results


In [ ]:
# Edit these values before running.
cfg = Config()
SCENARIO = "real"
N_PER_CLASS = 400
REAL_TRAIN_ROOT = ROOT / "data_final" / "train"
TEST_ROOT = ROOT / "data_final" / "test"
SYNTH_DIR = ROOT / "data_synth"
OUT_DIR = ROOT / "runs" / "clf"

cfg, SCENARIO, N_PER_CLASS


In [ ]:
single_result = run_classifier_notebook(
    cfg,
    SCENARIO,
    N_PER_CLASS,
    str(SYNTH_DIR),
    str(OUT_DIR),
    real_train_root=str(REAL_TRAIN_ROOT),
    test_root=str(TEST_ROOT),
)
single_result


In [ ]:
# Optional small grid.
GRID_SIZES = [100, 200]
GRID_SCENARIOS = ["real", "synth", "both", "real_aug"]
# grid_results = run_grid_notebook(cfg, GRID_SIZES, GRID_SCENARIOS, str(SYNTH_DIR), str(OUT_DIR), real_train_root=str(REAL_TRAIN_ROOT), test_root=str(TEST_ROOT))
